## 8: Building a Streamlit Frontend (OPTIONAL)

This notebook builds a customer-facing Streamlit web application with secure authentication, real-time chat, and session management. You'll launch the app and test the complete customer experience.

![Streamlit](images/Streamlit.png)

---

**Prerequisites:** Completed 4, agent deployed and running

### Step 1: Install Frontend Dependencies

In [4]:
# Install frontend-specific dependencies
%pip install -r utils/streamlit_frontend/requirements.txt -q
print("✅ Frontend dependencies installed successfully!")

Note: you may need to restart the kernel to use updated packages.
✅ Frontend dependencies installed successfully!


### Step 2: Understanding the Frontend Architecture

The application consists of:
- **main.py**: Streamlit UI and authentication
- **chat.py**: Chat management and Runtime integration
- **chat_utils.py**: Message formatting utilities (local development)
- **chat_utils_cloud.py**: Cloud-compatible utilities with Streamlit secrets support
- **sagemaker_helper.py**: URL generation helper
- **STREAMLIT_CLOUD_DEPLOYMENT.md**: Guide for deploying to Streamlit Cloud

### Step 2.5: Configure SSM Parameters

The Streamlit app needs the agent ARN stored in SSM Parameter Store. Let's set it up:

In [5]:
import os

# Set the AWS profile to match your shell environment
os.environ['AWS_PROFILE'] = 'workshop-profile'

import json
import boto3

# Get the notebook directory (in case we changed directories)
notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
# Navigate to parent directory if we're in streamlit_frontend
if notebook_dir.endswith('streamlit_frontend'):
    notebook_dir = os.path.dirname(os.path.dirname(notebook_dir))
elif notebook_dir.endswith('utils'):
    notebook_dir = os.path.dirname(notebook_dir)

config_path = os.path.join(notebook_dir, 'runtime_config.json')

# Load runtime configuration from 5
with open(config_path, 'r') as f:
    runtime_config = json.load(f)

agent_arn = runtime_config['agent_arn']

# Store agent ARN in SSM Parameter Store
ssm = boto3.client('ssm', region_name='us-west-2')

try:
    ssm.put_parameter(
        Name='/app/returnsrefunds/agentcore/runtime_arn',
        Value=agent_arn,
        Type='String',
        Overwrite=True,
        Description='Agent ARN for Returns and Refunds Streamlit app'
    )
    print(f"✅ Stored agent ARN in SSM Parameter Store")
    print(f"   Parameter: /app/returnsrefunds/agentcore/runtime_arn")
    print(f"   Value: {agent_arn}")
except Exception as e:
    print(f"❌ Error storing parameter: {e}")

✅ Stored agent ARN in SSM Parameter Store
   Parameter: /app/returnsrefunds/agentcore/runtime_arn
   Value: arn:aws:bedrock-agentcore:us-west-2:625579972148:runtime/returns_refunds_agent-HSGBtQFj7x


### Step 3: Launch the Application

Start the Streamlit server on port 8501. The app will connect to your deployed AgentCore Runtime.

**Note:** The app runs continuously until stopped (Ctrl+C). Cognito tokens are valid for 2 hours.

In [6]:
import subprocess
import time
import os
import sys
import platform

print("🚀 Starting Returns and Refunds Agent Streamlit Application...")
print("=" * 60)

# Check if streamlit is already running (cross-platform)
is_running = False
try:
    if platform.system() == 'Windows':
        # Windows: use tasklist
        result = subprocess.run(['tasklist', '/FI', 'IMAGENAME eq python.exe'], 
                              capture_output=True, text=True)
        is_running = 'streamlit' in result.stdout.lower()
    else:
        # Unix/Linux/Mac: use pgrep
        result = subprocess.run(['pgrep', '-f', 'streamlit'], capture_output=True)
        is_running = result.returncode == 0
except Exception as e:
    print(f"⚠️ Could not check if Streamlit is running: {e}")
    is_running = False

if is_running:
    print("\n✅ Streamlit appears to be already running!")
    print("   Access it at: http://localhost:8501")
else:
    print("\n🔄 Starting Streamlit app...")
    # Start streamlit in background
    original_dir = os.getcwd()
    os.chdir('utils/streamlit_frontend')
    
    # Start Streamlit process
    if platform.system() == 'Windows':
        # Windows: use CREATE_NEW_CONSOLE to run in background
        subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'main.py', 
                         '--server.port', '8501', '--server.address', '0.0.0.0', 
                         '--server.headless', 'true'],
                        creationflags=subprocess.CREATE_NEW_CONSOLE if platform.system() == 'Windows' else 0)
    else:
        # Unix/Linux/Mac
        subprocess.Popen(['streamlit', 'run', 'main.py', '--server.port', '8501', 
                         '--server.address', '0.0.0.0', '--server.headless', 'true'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    os.chdir(original_dir)
    time.sleep(5)
    print("   ✅ Streamlit started!")
    print("   Access it at: http://localhost:8501")

print("\n" + "=" * 60)

🚀 Starting Returns and Refunds Agent Streamlit Application...

🔄 Starting Streamlit app...
   ✅ Streamlit started!
   Access it at: http://localhost:8501



### Get the Application URL and Credentials

In [ ]:
# Display the Streamlit URL
print('\n🚀 Returns and Refunds Agent Streamlit Application URL:')
print('   http://localhost:8501\n')
print("Please use the following to login and test the Streamlit Application")
print("Username:       testuser")
print("Password:       MyPassword123!")

### Step 4: Testing Your Application

Once running, test the customer support experience:

1. Access the URL and sign in with test credentials
2. Verify the welcome message appears
3. Test return policy questions
4. Verify memory and context are maintained

### Sample Prompt to validate
- Return Policy Questions: I bought a Kindle Book three days ago by accident in India. I want to get a refund, what date is the ETA if I request it now?"

![Login Screen](images/streamlit_login.png)
![Agent Question](images/agent_question.png)

### Step 5: Customization Options

Customize by modifying:
- **main.py**: Colors, themes, branding, layout
- **chat.py**: Features, behavior, tool integration
- **chat_utils.py**: Messages, error handling, formatting

### Stopping the Application

To stop the Streamlit application:

In [ ]:
import subprocess
import platform

print("🛑 Stopping Streamlit application...")

try:
    if platform.system() == 'Windows':
        # Windows: use taskkill to stop streamlit processes
        result = subprocess.run(['taskkill', '/F', '/FI', 'WINDOWTITLE eq streamlit*'], 
                              capture_output=True, text=True)
        # Also try to kill by searching for streamlit in command line
        subprocess.run(['taskkill', '/F', '/IM', 'python.exe', '/FI', 'MEMUSAGE gt 0'], 
                      capture_output=True)
        print("✅ Streamlit application stopped (if it was running)")
        print("   Note: On Windows, you may need to manually close the console window")
    else:
        # Unix/Linux/Mac: use pkill
        result = subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
        if result.returncode == 0:
            print("✅ Streamlit application stopped")
        else:
            print("ℹ️ No Streamlit process found running")
except Exception as e:
    print(f"⚠️ Error stopping Streamlit: {e}")
    print("   You may need to manually stop the Streamlit process")
    if platform.system() == 'Windows':
        print("   Windows: Close the console window or use Task Manager")
    else:
        print("   Unix/Linux/Mac: Use 'pkill -f streamlit' in terminal")

### Summary

You've built a complete customer-facing application with secure authentication, real-time chat, and session management. The frontend seamlessly integrates with your deployed AgentCore Runtime.

### Next Steps

- Customize styling and branding
- Add multi-language support
- Integrate with CRM or ticketing systems
- Optimize for mobile devices